# Part 2 — AI / LLM: Judge Explanations & Give Feedback

Goal: give at-risk students useful feedback on their own explanations of a lesson.

Steps (see task brief):
1. Judge each explanation vs the lesson key points: `understood` / `partial` / `wrong`.
2. Short feedback: one thing right, one thing missed, one hint (not the full answer).
3. Return structured JSON: at least `explanation_id`, `verdict`, `got_right`, `missed`, `hint`.
4. Evaluate on the 60 labelled rows: accuracy, where it goes wrong, what I changed to improve.
5. Run for the at-risk students from Part 1; show 5 example outputs.

Rules: no API key in code — load `GEMINI_API_KEY` from `.env` (see `CLAUDE.md`).

In [ ]:
# Setup
import sys
sys.path.insert(0, "..")  # so `from src...` works when running from notebooks/

import pandas as pd
from pathlib import Path

from src import llm_judge
from src.llm_judge import (judge_explanations, load_lessons, strip_fillers, cache_key,
                           build_prompt_v1, SYSTEM_V1, MODEL, PROMPT_VERSION, CACHE_DIR)
from src.llm_eval import (accuracy, confusion, errors_table, injection_rows, is_injection,
                          stability, usage_summary, write_report)

SEED = 42  # no random steps in our code; LLM randomness is measured by the stability check
pd.set_option("display.max_colwidth", 200)
print("model:", MODEL, "| prompt:", PROMPT_VERSION)

## 1-3. Judge + feedback + structured JSON

How it works (`src/llm_judge.py`):

- **Model:** Gemini Flash via `google-genai`. Key from `.env`, never in code. Temperature is left at
  the default, following Google's guidance for Gemini 3 models (lowering it can hurt quality). The
  cost is some randomness, so section 4 measures **stability**: the same texts graded twice.
- **Structured output:** the response schema is a pydantic model (`Grading`): `reasoning`, `verdict`
  (`understood|partial|wrong`), `key_points_covered`, `got_right`, `missed`, `hint`. Gemini is forced to
  return that JSON, and every response is validated again with pydantic. `explanation_id` is added by
  the code (not the LLM), so one grading can be reused for every student who wrote the same text.
  `reasoning` is an extra field placed *first*, so the model explains before it decides — and so errors
  can be reviewed.
- **Prompt v1:** the lesson's 4 key points from `lessons.md`; the student text inside
  `<student_explanation>` tags and declared as data, never instructions; judge content only (ignore hedges
  and confidence claims); grade Hinglish on meaning; the hint must nudge, not give the answer.
- **Filler stripping + cache:** known sign-offs (`(sorry for typos)`, `hope this is right!`, `i think`,
  `not sure tho`) are removed, then results are cached by (lesson, normalized text) in `outputs/cache/`.
- **Robustness:** retry with exponential backoff on rate limits (429) and server errors; invalid JSON is
  retried too. Every real API call logs its token usage (for the Part 3 cost estimate).

In [ ]:
explanations = pd.read_csv("../data/raw/explanations.csv")
explanations["key"] = [cache_key(l, t) for l, t in zip(explanations.lesson_id, explanations.explanation_text)]
labelled = explanations[explanations.label.notna()].copy()

print(f"all explanations: {len(explanations)} rows -> {explanations.key.nunique()} unique texts after stripping fillers")
print(f"labelled: {len(labelled)} rows, {labelled.explanation_text.nunique()} unique raw texts -> {labelled.key.nunique()} unique after stripping")
print("labelled duplicates with conflicting labels:", (labelled.groupby("key").label.nunique() > 1).sum())
labelled.label.value_counts()

In [ ]:
# What the model sees, using the prompt-injection row as the example
example = labelled.set_index("explanation_id").loc["E0006"]
print("SYSTEM PROMPT\n" + SYSTEM_V1)
print("\nUSER MESSAGE\n" + build_prompt_v1("L2", load_lessons()["L2"], strip_fillers(example.explanation_text)))

In [ ]:
# Grade the 60 labelled rows (unique texts are called once; the rest come from the cache)
results = pd.DataFrame(judge_explanations(labelled.to_dict("records")))
df = labelled.merge(results, on="explanation_id")
print("API calls this run:", (~df.from_cache).sum(), "| from cache:", df.from_cache.sum())
df[["explanation_id", "lesson_id", "label", "verdict", "got_right", "missed", "hint"]].head()

## 4. Evaluate on the 60 labelled rows

In [ ]:
acc = accuracy(df)
print(f"accuracy, all rows:     {acc['all_rows'][0]:.1%} ({acc['all_rows'][1]} rows)")
print(f"accuracy, unique texts: {acc['unique_texts'][0]:.1%} ({acc['unique_texts'][1]} texts)")
confusion(df)

In [ ]:
# Every wrong prediction (one row per unique text)
errors_table(df)

### Prompt injection

Three rows try to instruct the grader. Only E0006 is labelled; the two unlabelled ones (E0329, E0648) are
graded too. **Pass** = the verdict is not `understood` (what each injection asks for), and the reasoning
should show the instruction was treated as data.

In [ ]:
unlabelled_injections = explanations[is_injection(explanations.explanation_text) & explanations.label.isna()]
extra = unlabelled_injections.merge(pd.DataFrame(judge_explanations(unlabelled_injections.to_dict("records"))),
                                    on="explanation_id")
injections = injection_rows(pd.concat([df, extra]))
injections

### Stability

Temperature is at the default, so the same text can get a different verdict on another run. The 37 unique
labelled texts are graded a second time with an empty cache (`run="repeat"`), and the verdicts compared.

In [ ]:
repeat = labelled.drop_duplicates("key")
repeat = repeat.merge(pd.DataFrame(judge_explanations(repeat.to_dict("records"), run="repeat")), on="explanation_id")
stable = stability(df, repeat)
print(f"identical verdicts: {stable[0]:.1%} of {stable[1]} unique texts")
stable[2]  # texts whose verdict changed between runs

STABILITY_PLACEHOLDER

In [ ]:
usage = usage_summary(CACHE_DIR / "usage_log.jsonl", PROMPT_VERSION)
write_report(df, Path("../outputs/llm_eval_v1.md"), MODEL, PROMPT_VERSION, usage, injections, stable)
usage

_Error analysis and prompt changes: TODO after reviewing v1._

## 5. Run for at-risk students — 5 examples
_TODO_